## Importación de librerías necesarias

In [1]:
%load_ext IPython.extensions.autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
import sys

def find_src_folder(current_path: Path, folder_name: str = 'src') -> Path:
    search_directories = [current_path] + list(current_path.parents)
    for parent in search_directories:
        if parent.name == folder_name:
            return parent.parent
    return current_path

src_path = find_src_folder(Path.cwd(), 'src')
sys.path.append(str(src_path))

print(f"JAVA_HOME: {os.environ.get('JAVA_HOME')}")
print(f"TFHUB_CACHE_DIR: {os.environ.get('TFHUB_CACHE_DIR')}")

JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
TFHUB_CACHE_DIR: /mnt/d/Maestría/Amazon Reviews Code/tf_cache


In [3]:
from pyspark.sql import functions as F, types as T, DataFrame, Window
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, fowlkes_mallows_score
from sklearn.cluster import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.clustering import KMeans, KMeansModel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import array_to_vector
from src.gold.training.pca import PCAEncoder

In [4]:
import sys
from pathlib import Path
def find_src_folder(current_path: Path, folder_name: str = 'src') -> Path:
    search_directories = [current_path] + list(current_path.parents)
    for parent in search_directories:
        if parent.name == folder_name:
            return parent.parent
    return current_path
src_path = find_src_folder(Path.cwd(), 'src')
sys.path.append(str(src_path))

In [5]:
from src.utils.spark import SparkUtils
spark_utils = SparkUtils('predict_flow')
spark = spark_utils.spark

:: loading settings :: url = jar:file:/mnt/d/Maestr%c3%ada/Amazon%20Reviews%20Code/.venv-linux/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/edgar/.ivy2/cache
The jars for the packages stored in: /home/edgar/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7731cadf-8a32-48e3-819b-e4cc269cc85f;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 116ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   

In [6]:
GOLD_SCHEMA = 'gold.premodeling'
GOLD_SCHEMA_CLUSTER = 'gold.cluster'
GOLD_SCHEMA_ENCODING = 'gold.encoding'
SCHEMA = 'silver.preprocess'

## Importar información de referencia

In [21]:
meta_items = spark.read.format('delta').load(spark_utils.path('meta_items', 'bronze'))

reviews_indexed = spark.read.format('delta').load(spark_utils.path(
    'reviews_indexed', catalog = SCHEMA
))

df_vectorized_pca_phrases_with_index = spark.read.format('delta').load(spark_utils.path(
    'df_vectorized_pca_phrases_with_index', catalog = 'gold.premodeling'
))

In [8]:
ASIN = "1610121147"

## Recibir entrada de usuario separada por componente

In [7]:
TITLE = "Turning Technologies QT Device RCQR-01"
DESCRIPTION = ['Large LCD Screen & Backlight', 'Full QWERTY Keyboard', 'Self-Paced Testing Mode']
FEATURES = ['Compatible with Samsung Galaxy A02 / Samsung Galaxy M02.', 'Military grade drop protection armor case with metal belt clip built-in kickstand, high quality and easy to carry.', 'Built-in kickstand stand feature for hands-free movie watching, videos, pictures and more, all ports, buttons and speakers have precision cut-outs for easy access.', 'Fits magnetic car mount holder, magnetic metal plate can be attracted to a magnetic car mount holder, [car mount holder NOT included].', 'Combination of soft TPU and hard Polycarbonate bumper, military grade drop protection case for your phone against any scratch, bump, finger marks, and dirt.']

In [ ]:
from src.utils.testers.FullTester import FullTester
full_tester = FullTester(
    spark=spark,
    spark_utils=spark_utils,
)

full_tester.set_components({
    "title": TITLE,
    "description": DESCRIPTION,
    "features": FEATURES,
})

full_tester.clean_components()
full_tester.separate_sentences_per_component()
full_tester.encode_sentences()
full_tester.summarize_sentences_by_component()
full_tester.pca_encode()
full_tester.find_pairs()
full_tester.build_models_inputs()
full_tester.generate_prediction()

04:06:15.370 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


04:06:21.050 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null
04:06:25.432 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null
04:06:29.293 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null
04:06:32.569 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


(DataFrame[entity_id: int, title: string, sentence_number: int, record_id: bigint],
 DataFrame[entity_id: int, description_sentence: string, paragraph_number: int, sentence_number: int, record_id: bigint],
 DataFrame[entity_id: int, feature_sentence: string, feature_number: int, sentence_number: int, record_id: bigint])

In [ ]:
from src.utils.models.clustering.LSHNeighborsClustering import LSHNeighborsClustering,LSHNeighborsClusteringParameter


LSHNeighborsClustering(
            lsh_parameters=LSHNeighborsClusteringParameter(
                comb_id="pca_features_1000_300",
                n_neighbors=10,
                min_cosine_similarity=0.2,
                bucket_length=1.0,
                num_hash_tables=5
            ),
            spark_utils=spark_utils,
            table_prefix="lsh_pca_features",
            catalog=self.cluster_schema,
            id_col="entity_id",
            features_col="features",
            hashes_col="hashes",
            batch_size=10_000
        )

In [ ]:
full_tester.df_vectorized_pca.count()

456530

In [ ]:
full_tester.title_sentences_df.show()

+---------+--------------------+---------------+---------+
|entity_id|               title|sentence_number|record_id|
+---------+--------------------+---------------+---------+
|        1|Turning Technolog...|              1|        0|
+---------+--------------------+---------------+---------+



In [ ]:
full_tester.features_sentences_df.show()

+---------+--------------------+--------------+---------------+---------+
|entity_id|    feature_sentence|feature_number|sentence_number|record_id|
+---------+--------------------+--------------+---------------+---------+
|        1|Compatible with S...|             1|              1|        0|
|        1|Military grade dr...|             2|              1|        1|
|        1|Built-in kickstan...|             3|              1|        2|
|        1|Fits magnetic car...|             4|              1|        3|
|        1|Combination of so...|             5|              1|        4|
+---------+--------------------+--------------+---------------+---------+



In [ ]:
meta_items_features_text_clean = spark.read.format('delta').load(spark_utils.path(
    'meta_items_features_text_clean', catalog = 'gold.premodeling'
))

In [ ]:
ASIN

'1610121147'

In [ ]:
meta_items_features_text_clean.filter(F.col('parent_asin') == ASIN).show()

+-----------+--------------------+--------------+---------------+---------+
|parent_asin|    feature_sentence|feature_number|sentence_number|record_id|
+-----------+--------------------+--------------+---------------+---------+
| 1610121147|Brand New Materia...|             1|              1|      217|
| 1610121147|Keep your iphone ...|             2|              1|      218|
| 1610121147|Made of aluminum,...|             3|              1|      219|
| 1610121147|t is light and du...|             4|              1|      220|
| 1610121147|Unique design all...|             5|              1|      221|
+-----------+--------------------+--------------+---------------+---------+



In [ ]:
meta_items_features_text_clean.filter(F.col('parent_asin') == ASIN).limit(1).toPandas()

,parent_asin,feature_sentence,feature_number,sentence_number,record_id
0,1610121147,Brand New Material Aluminum Alloy Color:,1,1,217


In [ ]:
full_tester.features_sentences_encoded_df.show()

+---------+--------------------+
|record_id|     text_embeddings|
+---------+--------------------+
|        0|[-0.0334079600870...|
|        1|[-0.0446165613830...|
+---------+--------------------+



In [ ]:
full_tester.df_vectorized_pca.show()

+----------+--------------------+
| entity_id|            features|
+----------+--------------------+
|1934931519|[0.33852120009204...|
|193500932X|[0.31975890058713...|
|8862931379|[0.45299272512449...|
|9861057021|[0.39073514436394...|
|B00000J1QR|[0.44048907160245...|
|B00000JD4T|[0.62266552363724...|
|B00000K4BB|[0.43766026423951...|
|B00001RMD3|[0.50443085315336...|
|B000021YTH|[0.51511199016236...|
|B00002JXFH|[0.61228884894236...|
|B00004SYN4|[0.53972380111307...|
|B00004SZ9F|[0.52067111979480...|
|B00004THDC|[0.51860509812582...|
|B00004X0ZH|[0.52762458695420...|
|B0000513W5|[0.57722568870434...|
|B000054P4C|[0.69156594243872...|
|B00005AY8G|[0.62746377980514...|
|B00005JM2X|[0.35835799903836...|
|B00005LEOC|[0.51646184693192...|
|B00005LKXR|[0.39127036593490...|
+----------+--------------------+
only showing top 20 rows



In [ ]:
full_tester.similar_pairs_df.show(10)

+----------+--------------------+-------------------+
| entity_id|            features|         cosine_sim|
+----------+--------------------+-------------------+
|1400501466|[0.48793111617572...|0.40950027585029825|
|0110400550|[0.31313488074614...| 0.3320543841189014|
|1400532736|[0.61795149272006...|0.44187590398092325|
|1935009311|[0.33676542833642...| 0.3684678818499538|
|1948701847|[0.50234143086782...|0.30361258818201275|
|1935009338|[0.32036687159697...|0.28999317275352077|
|0594450268|[0.41613413426319...| 0.4715018488522106|
|0972683275|[0.45232116204421...| 0.2940264823416773|
|1935009362|[0.46957275598499...| 0.3318842042819826|
|1400501741|[0.32378757964099...| 0.2483499062694574|
+----------+--------------------+-------------------+
only showing top 10 rows



## Pruebas agrupación

### Ejecución hallazgo de vecinos mas cercanos

In [10]:
MIN_COSINE_SIMMILARITY = 0.9
NUM_NEIGHBORS = 10

In [12]:
from src.utils.models.clustering.LSHNeighborsClustering import LSHNeighborsClustering, LSHNeighborsClusteringParameter

lsh_clustering = LSHNeighborsClustering(
    lsh_parameters=LSHNeighborsClusteringParameter(
        comb_id="pca_features_1000_300",
        n_neighbors=NUM_NEIGHBORS,
        min_cosine_similarity=MIN_COSINE_SIMMILARITY,
        bucket_length=1.0,
        num_hash_tables=5
    ),
    spark_utils=spark_utils,
    table_prefix="lsh_pca_features_balanced",
    catalog=GOLD_SCHEMA_CLUSTER,
    id_col="parent_asin",
    features_col="features",
    hashes_col="hashes",
    batch_size=10_000
)

In [13]:
pairs = spark.read.format('delta').load( lsh_clustering.pairs_path )


In [14]:
pairs.count()

20:18:49.941 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


69357

In [15]:
pairs.filter(F.col('parent_asin_a') == "B074JLV5XX").show(20)

+-------------+-------------+------------------+
|parent_asin_a|parent_asin_b|        cosine_sim|
+-------------+-------------+------------------+
|   B074JLV5XX|   B075ZDFYDY|0.9411782838076508|
|   B074JLV5XX|   B07Q5VKF1G|0.9462786325966811|
|   B074JLV5XX|   B078B9RGR8| 0.952629702696235|
|   B074JLV5XX|   B07PLMT6Y2| 0.939043889033465|
|   B074JLV5XX|   B08B9XFWT1|0.9045908933375996|
|   B074JLV5XX|   B08G8ZFVLM| 0.952977954014717|
|   B074JLV5XX|   B07VWSDPSN|0.9428793011636314|
|   B074JLV5XX|   B07VZWJGS7|0.9062022470757471|
|   B074JLV5XX|   B08GF1HZKJ|0.9465701004643476|
|   B074JLV5XX|   B08713GYDZ| 0.929996829606414|
|   B074JLV5XX|   B081B9CQ46| 0.943887666910909|
+-------------+-------------+------------------+



In [19]:
meta_items.filter(F.col('parent_asin') == "B074JLV5XX").toPandas().to_dict()

{'title': {0: 'VESA Mount Adapter for Dell S2218, S2318, S2319, S2418, S2419, S2718, S2719 Monitors | Does Not Fit Ultrathin Monitors S2718D, S2719HM, and S2719DM | [Patented] - by HumanCentric'},
 'main_category': {0: 'Office Products'},
 'features': {0: ['FITS SELECT DELL S-SERIES MONITORS: This bracket is tested and guaranteed to work with: Dell S2218H, S2318HX, S2318HN, S2318NX, S2319H, S2319HN, S2418HX, S2418HN, S2418NX, S2419H, S2419HN, S2718HX, S2718HN, S2718NX, S2719H, and S2719HN (Does not fit any Ultrathin monitors such as S2718D, S2419HM and S2719DM)',
   'IMPORTANT NOTE: Does not fit the SE2219H, SE2419H, and SE2719H - Please select the size labeled "Dell S2*18, S2*19, and SE2*19 (Not Ultrathin Models)" for the VESA adapter for those model numbers',
   'MOUNT YOUR DELL MONITOR ON A STANDARD VESA MOUNT - Even though these monitors weren’t made with mounting holes, our convenient bracket enables you to connect your Dell monitor anyway! Whether you’re looking to mount it on th

In [26]:
meta_items.filter(F.col('parent_asin') == "B081B9CQ46").toPandas().to_dict()

{'title': {0: 'HumanCentric VESA Mount Adapter for Viewsonic Models VX2776-smhd, VX2476-smhd, VX2376-smhd, and VX2276-smhd'},
 'main_category': {0: 'Computers'},
 'features': {0: ['FITS VIEWSONIC MONITORS: This bracket is only tested and guaranteed to work with the following models: Viewsonic VX2776-smhd, VX2476-smhd, VX2376-smhd, and VX2276-smhd',
   'MOUNT YOUR VIEWSONIC MONITOR ON A STANDARD VESA MOUNT - Even though these monitors weren’t made with mounting holes, our convenient bracket enables you to connect your Viewsonic monitor anyway! Whether you’re looking to mount it on the wall or just get it off your desk, this bracket will let you connect your monitor in seconds. Works with double and triple (and more) mounts too!',
   'EASY INSTALLATION - You’ll be able to install this bracket in less than two minutes with nothing more than a screwdriver! Just remove the existing base and snap ours in place instead - then connect it right to the VESA mount. We’ve tested our mount extensiv

In [29]:
a = reviews_indexed.filter(F.col('parent_asin') == "B081B9CQ46").toPandas()

In [30]:
a.values

array([['Works perfectly',
        "If you're like me and got monitors from work so you could work from home, you got View Sonics. This conversion kit works perfectly and allowed me to attach both of my monitors to a VESA wall-mount. The price isn't bad for the quality metal construction and no non-sense approach.",
        'B081B9CQ46', 5.0, 1, 1, 2654104],
       ['Great for ultra thin monitors',
        'I was upset to find out the viewsonic monitors I bought didn’t have a vesa mount. So I ordered this and it attached to the vesa mount perfectly.',
        'B081B9CQ46', 5.0, 1, 1, 2654102],
       ['Happy with it',
        "It works fine. I'm just not sure why it they made it so it sticks out so far back from the monitor? It should be more flush.",
        'B081B9CQ46', 4.0, 0, 1, 2654112],
       ['A mount solution for',
        "The stabalizer doesn't seem to add much stability when monitor is vertical. But I dont notice it making much of a difference. otherwise everything else is

In [ ]:
(
    pairs.groupBy('parent_asin_a')
        .agg(F.count('*').alias('count'))
        .filter(F.col('count') > 10)
        .orderBy('count', ascending=False).show()
)

+-------------+-----+
|parent_asin_a|count|
+-------------+-----+
|   B09JNWHXWW|  132|
|   B008I65B8Y|  110|
|   B008I6430Q|  110|
|   B008I63BZ4|  110|
|   B008I64UX6|  110|
|   B00FBIZR4U|  110|
|   B005STXRPI|  110|
|   B00FBJ4G82|  110|
|   B00FBJA6QS|  110|
|   B008I62AYC|  110|
|   B008I639UQ|  110|
|   B008I641KS|  110|
|   B008I64NB0|  110|
|   B008I64MAC|  110|
|   B008I63C9O|  110|
|   B00FBJ460K|  110|
|   B008I63K00|  110|
|   B008I63CTO|  110|
|   B008I63JS8|  110|
|   B008I63REY|  110|
+-------------+-----+
only showing top 20 rows



In [ ]:
meta_items.filter(F.col('main_category') == F.lit('Computers')).count()

444322

In [ ]:
pairs.filter(F.col('parent_asin_a') == "B09JNWHXWW").show()

+-------------+-------------+------------------+
|parent_asin_a|parent_asin_b|        cosine_sim|
+-------------+-------------+------------------+
|   B09JNWHXWW|   B09GG44PXR| 0.931829471970991|
|   B09JNWHXWW|   B0B743QR7T|0.9051063748384647|
|   B09JNWHXWW|   B09J2BMSSL|0.9029107100138709|
|   B09JNWHXWW|   B09WRF3C8K|0.9441044620580835|
|   B09JNWHXWW|   B09NVJKK3B|0.9055025750666184|
|   B09JNWHXWW|   B09Y63XNR3|0.9053854904992825|
|   B09JNWHXWW|   B098N8PBWG|0.9096501620387906|
|   B09JNWHXWW|   B0B14LH7BS|0.9047262525203075|
|   B09JNWHXWW|   B0B41J6HK3|0.9202711952666911|
|   B09JNWHXWW|   B09DGDVDQL|0.9275403241681206|
|   B09JNWHXWW|   B0BG5N2BP1|0.9177163646093404|
|   B09JNWHXWW|   B09ZFB8SL5|0.9150768252382715|
|   B09JNWHXWW|   B0B3VQF33X|0.9220218181668001|
|   B09JNWHXWW|   B09MYN51N5|0.9132612446884449|
|   B09JNWHXWW|   B09SLTR62D| 0.951469286141775|
|   B09JNWHXWW|   B09S5ML6WC|0.9363251265907286|
|   B09JNWHXWW|   B0924M5YTR|0.9160840482268032|
|   B09JNWHXWW|   B0

In [ ]:
a = reviews_indexed.filter(F.col('parent_asin') == "B0BG715FY1").toPandas()

In [ ]:
len(a)

24

In [ ]:
meta_items.filter(F.col('parent_asin') == "B0BG715FY1").toPandas().to_dict()

{'title': {0: 'Apple MFi Certified iPhone Charger,2 Set 3.3FT Lightning Cable with USB Plug Fast Charging Cube High Speed Data Sync USB Cable Compatible with iPhone 11/12/13 Pro Max/XS/XR/X/8/7/Plus/6S/SE/iPad'},
 'main_category': {0: 'Industrial & Scientific'},
 'features': {0: ['[Safe & Fast Charger] iPhone charger set approved by the U.S. Safe Market. This iPhone charger brick Multiple built-in safeguards and intelligent IC identification technology protect against short circuit, over-current, over-voltage, over-heating and over-charging.',
   '[2 Sets iPhone Charger Cube with Cord] 2 Pack iPhone wall charger block for your convenience, Lightweight, compact design that fits your storage requirements. You can take it when you travel, The iPhone Charger is ideal for charging at home or in the office or travel outside, which is lightweight in weight and small in size.',
   '[Apple Certified Lightning Cable] MFi Certified iPhone Lightning Charger Cord built in an original chip, which me

In [ ]:
meta_items.filter(F.col('parent_asin') == "B0B41J6HK3").toPandas().to_dict()

{'title': {0: 'OuTrade iPhone Charger Cable, 3 Pack MFi Lightning Cable Durable Braided Fast Charging Cord Compatible for iPhone SE 12 11 11 Pro 11 Pro Max Xs MAX XR X 8 7 6S 6, iPad and More (Black, 10 ft)'},
 'main_category': {0: 'Industrial & Scientific'},
 'features': {0: ['【MFi Certified Lightning Cable】 Built with original MFi chip promise no warning message pops up, ensured safe fast charging and data transfer for your devices',
   '【2.4A Fast Charging & 480Mbps Data Transfer】 High-quality four-core copper wires enhance charging & data transfer speed of the iPhone cables. Built-In MFi Certified chipset ensures a faster charging time while keeping your device completely safe',
   '【Premium Durable Braided Material】 iPhone cord with laser welding technology that have passed 50,000+ times bending tests for extra protection and durability. Braided insulation and precisely layer-welded connectors, which make cable more durable and sturdier than normal iPhone charger cables but also f

In [ ]:
main_category_encoded.columns

['title',
 'main_category',
 'features',
 'description',
 'average_rating',
 'rating_number',
 'price',
 'store',
 'parent_asin',
 'categories',
 'details',
 'images',
 'description_colapsed',
 'features_colapsed',
 'colapsed_text',
 'colapsed_text_spelled',
 'colapsed_text_words',
 'colapsed_text_length',
 'review_count',
 'main_category_computers',
 'main_category_all_electronics',
 'main_category_home_audio_theater',
 'main_category_amazon_home',
 'main_category_industrial_scientific',
 'main_category_cell_phones_accessories',
 'main_category_car_electronics',
 'main_category_office_products',
 'main_category_camera_photo']

### Ejecución hallazgo de vecinos mas cercanos (miniLLM)

In [9]:
MIN_COSINE_SIMMILARITY = 0.9
NUM_NEIGHBORS = 10

In [10]:
from src.utils.models.clustering.LSHNeighborsClustering import LSHNeighborsClustering, LSHNeighborsClusteringParameter

lsh_clustering_minillm = LSHNeighborsClustering(
    lsh_parameters=LSHNeighborsClusteringParameter(
        comb_id="pca_features_1000_300",
        n_neighbors=10,
        min_cosine_similarity=MIN_COSINE_SIMMILARITY,
        bucket_length=1.0,
        num_hash_tables=5
    ),
    spark_utils=spark_utils,
    table_prefix="lsh_pca_features_minillm",
    catalog=GOLD_SCHEMA_CLUSTER,
    id_col="parent_asin",
    features_col="features",
    hashes_col="hashes",
    batch_size=10_000
)

In [11]:
pairs = spark.read.format('delta').load(spark_utils.path(
    'lsh_pairs_lsh_pca_features_minillm_pca_features_1000_300', catalog=GOLD_SCHEMA_CLUSTER
))


In [ ]:
pairs_minillm = pairs

In [ ]:
pairs_minillm.filter(F.col('parent_asin_a') == "B000NNS5XS").count()

1

In [ ]:
pairs_use.filter(F.col('parent_asin_a') == "B000NNS5XS").count()

11:12:23.690 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


12

In [ ]:
pairs_use.alias('A').join(
    pairs_minillm.alias('B'), on='parent_asin_a', how='inner'
).select('A.parent_asin_a').distinct().show()

+-------------+
|parent_asin_a|
+-------------+
|   B0011FTKFY|
|   B00123V6PW|
|   B000H7GVA4|
|   B000UTIB7A|
|   B000NNS5XS|
|   B000UHDRAS|
|   B000VEPC6W|
|   B000X1OGGU|
|   B000IEAYN6|
|   B000EZKAHO|
|   B000ZJQO6A|
|   B000VABPAI|
|   B0012MULLI|
|   B000II0AKO|
|   B000SDW84K|
|   B000N8HX90|
|   B00126MAHW|
|   B000WWQJ88|
|   B000NOUP0S|
|   B000IJKHU6|
+-------------+
only showing top 20 rows



In [ ]:
(
    pairs.groupBy('parent_asin_a')
        .agg(F.count('*').alias('count'))
        .filter(F.col('count') > 10)
        .orderBy('count', ascending=True).show()
)

+-------------+-----+
|parent_asin_a|count|
+-------------+-----+
|   B000GX47R8|   11|
|   B000ELG8PG|   11|
|   B00113V748|   11|
|   B000O8WE66|   11|
|   B000XRQ2Q6|   11|
|   B000YE8SAQ|   11|
|   B000SR0690|   11|
|   B0012OI6HW|   11|
|   B000V1SUQE|   11|
|   B000IEFILY|   11|
|   B000SNP0RW|   11|
|   B000F1WBUG|   11|
|   B000ENOSQ0|   11|
|   B000EYFFDY|   11|
|   B000W8U4GK|   11|
|   B000W9DO7U|   11|
|   B000LZCIXQ|   11|
|   B000EP5MQI|   11|
|   B000CDA7US|   11|
|   B0012OKAP8|   11|
+-------------+-----+
only showing top 20 rows



In [ ]:
pairs_use.filter(F.col('parent_asin_a') == "B000NNS5XS").show()

+-------------+-------------+------------------+
|parent_asin_a|parent_asin_b|        cosine_sim|
+-------------+-------------+------------------+
|   B000NNS5XS|   B0011ZCDKS| 0.918802649577544|
|   B000NNS5XS|   B000V1XLNG|0.9217574779281436|
|   B000NNS5XS|   B000HAOVGM|0.9240184806842342|
|   B000NNS5XS|   B000Q30420|0.9141726918912301|
|   B000NNS5XS|   B000OSLLPG|0.9801362033890302|
|   B000NNS5XS|   B000V1VG2E|0.9456833395616066|
|   B000NNS5XS|   B001IKKDKS|0.9044325495609654|
|   B000NNS5XS|   B001T9NUQM|0.9027998741503909|
|   B000NNS5XS|   B001G60DXG|0.9003039326496802|
|   B000NNS5XS|   B0012YA4YK|0.9023817854068865|
|   B000NNS5XS|   B001SER4AG|0.9012627772070566|
|   B000NNS5XS|   B001G60DWM|0.9053921046899697|
+-------------+-------------+------------------+



In [ ]:
pairs_minillm.filter(F.col('parent_asin_a') == "B000NNS5XS").show()

+-------------+-------------+------------------+
|parent_asin_a|parent_asin_b|        cosine_sim|
+-------------+-------------+------------------+
|   B000NNS5XS|   B0036RH3T0|0.9357998889528806|
+-------------+-------------+------------------+



In [ ]:
meta_items.filter(F.col('parent_asin') == "B000NNS5XS").toPandas().to_dict()

{'title': {0: 'Canon PowerShot SD1000 7.1MP Digital Elph Camera with 3x Optical Zoom (Black) (OLD MODEL)'},
 'main_category': {0: 'Camera & Photo'},
 'features': {0: ['7.1-megapixel CCD captures enough detail for photo-quality 15 x 20-inch prints',
   '3x optical zoom; ISO 1600 and High ISO Auto',
   'DIGIC III Image Processor; Face Detection AF/AE',
   'Selectable shooting modes and special scene modes',
   'Print/Share button makes direct printing simple']},
 'description': {0: ['Product Description',
   'Canon PowerShot SD1000 7.1MP Digital Elph Camera with 3x Optical Zoom (Black)',
   'From the Manufacturer',
   'Canon looked to the very first Elph for inspiration when designing the PowerShot SD1000 Digital Elph, and came up with a quintessential iteration of the icon: slim, clean-lined and fully flat. Inside, the SD1000 Digital Elph looks only to the future: 7.1 megapixels, a 3x optical zoom and advanced DIGIC III ensure top-quality images, while focus is fast and sharp and red-ey

In [ ]:
meta_items.filter(F.col('parent_asin') == "B0036RH3T0").toPandas().to_dict()

{'title': {0: 'NETGEAR Digital Entertainer Express (EVA9100)'},
 'main_category': {0: 'All Electronics'},
 'features': {0: ['Plays video, MP3s and digital photos from your PC, Mac or network attached storage device (NAS) to your HDTV',
   'Browse and play popular movies and TV shows from Hulu, Netflix, CBS, and more using PlayOn on your PC (free trial and special discountº)',
   'Automatically finds all digital media files on your home network and organizes them into an easily accessible library',
   'Includes one HDMI port, HD upconversion up to 1080p, and two USB ports for iPod and USB drives',
   'Easily add a NETGEAR Wireless USB Adapter (EVAW111) to connect wirelessly']},
 'description': {0: ['Product Description',
   'Why NETGEAR’s Digital Entertainer Express (EVA9100)? Plays video, MP3s and digital photos from your PC, Mac or network attached storage device (NAS) to your HDTV, Browse and play popular movies and TV shows from Hulu, Netflix, CBS, and more using PlayOn on your PC (